# TabM — Optuna HPO on T4 x2, fe_v4_native

Tunes **TabM** (`pytabkit.TabM_D_Classifier`) on the churn dataset using Optuna TPE,
then refits the winner as a 2-seed deep ensemble across both T4s — the same setup as
`kaggle/predict-customer-churn-tabm-gpu-min3.ipynb` but with learned hyperparameters
replacing library defaults.

**Parent run:** `20260601-022539-10bd38` (tabm-baseline, fe_v0, default params,
OOF ROC-AUC 0.9136).

**Search space** (3 dimensions, TPE):
- `lr` — initial learning rate, log-uniform [1e-4, 1e-2]
- `weight_decay` — L2 regularisation, log-uniform [1e-6, 1e-1]
- `batch_size` — mini-batch size, categorical {256, 512, 1024, 2048}

Architecture and PLR (piece-wise linear representation) settings are held fixed at
`TabM_D_Classifier` defaults — the `_D` calibration already tuned those. This study
targets the training-dynamics knobs that the defaults do not adapt per-dataset.

**Inner CV.** `StratifiedKFold(3)` on a stratified **120k-row subsample** drawn once
(fixed seed) so all trials are evaluated on identical rows. Each trial therefore
takes ~5–15 min instead of the ~78 min a full-data fold costs.

**12-hour budget management.** `NOTEBOOK_T0 = time.time()` is set at clone time;
Optuna's `timeout` parameter stops the study with enough margin for the final
5-fold 2-seed refit (~2.5 h). A `SaveBestParamsCallback` writes
`/kaggle/working/best_params.json` after every improvement — if the notebook is
hard-killed mid-trial the last-saved JSON holds the best params seen so far.

**Fold contract.** `StratifiedKFold(5, shuffle, seed 42)` is asserted against
`experiments/cv_folds_seed42.csv.gz` so the OOF is stackable against all local runs.
**Push that file to GitHub master before running.**

**Settings (right sidebar).** Accelerator → **GPU T4 x2** (not P100); Internet →
**On**; Add Input → **playground-series-s6e3**.

> ⏱️ Timing: study ~9 h (target ~40–60 trials at ~5–15 min/trial); refit ~1.5–2.5 h.
> Total ≤ 12 h via the timeout guard.
> **Smoke-test first:** set `SMOKE_TEST = True` in the time-budget cell, confirm
> one trial runs end-to-end and both GPUs light up in `nvidia-smi`, then restore
> and Save & Run All.

In [1]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import os, sys, subprocess, time

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
print("CWD:", os.getcwd())

# Wall-clock reference used by the 12-hour budget guard.
NOTEBOOK_T0 = time.time()

Cloning into '/kaggle/working/Predict-Customer-Churn'...
Updating files: 100% (319/319), done.


CWD: /kaggle/working/Predict-Customer-Churn


In [3]:
!pip install -q pytabkit optuna    # Kaggle already ships torch + CUDA

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())     # expect 2 on T4 x2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12

In [4]:
# cuda.is_available() can return True on an incompatible GPU - verify with a real op.
# Both devices must be healthy: the inner-CV study uses cuda:0; the 2-seed refit
# uses both.
import torch
for d in range(torch.cuda.device_count()):
    try:
        _ = (torch.randn(16, device=f"cuda:{d}") @ torch.randn(16, 16, device=f"cuda:{d}")).sum().item()
        print(f"cuda:{d} ({torch.cuda.get_device_name(d)}): compute OK")
    except Exception as e:
        print(f"cuda:{d} compute FAILED:", e)    # switch to T4 x2 and restart

from pytabkit import TabM_D_Classifier
print("TabM_D_Classifier imported OK")

cuda:0 (Tesla T4): compute OK
cuda:1 (Tesla T4): compute OK
TabM_D_Classifier imported OK


In [5]:
# data/processed/*.parquet are git-ignored - absent from the clone.
# Rebuild NATIVE (category-dtype) frames: pytabkit consumes category columns directly.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(encoding='native', force=True)
print(f'Loaded native: train_df {train_df.shape}, test_df {test_df.shape}')

Preprocessed and saved (native): train_df (594194, 21), test_df (254655, 20)
Loaded native: train_df (594194, 21), test_df (254655, 20)


### Feature engineering — fe_v4_native (minimal set)

Same `engineer_features` as the GBDT min3 and tabm-gpu-min3 runs: `AverageMonthly`
plus the two low-cardinality crosses, all row-wise (stateless). No `tenure == 0` rows
exist in the data, so `AverageMonthly` is always finite — pytabkit's preprocessing
rejects infinities.

In [6]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from src.tracking import DATA_DIR

DATA_VERSION = 'fe_v4_native'


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['AverageMonthly'] = df['TotalCharges'] / df['tenure']
    df['contract_x_payment'] = (df['Contract'].astype(str) + ' | '
                                + df['PaymentMethod'].astype(str)).astype('category')
    df['contract_x_internet'] = (df['Contract'].astype(str) + ' | '
                                  + df['InternetService'].astype(str)).astype('category')
    return df


fe_train_path = DATA_DIR / f'train_df_{DATA_VERSION}.parquet'
fe_test_path  = DATA_DIR / f'test_df_{DATA_VERSION}.parquet'

if fe_train_path.exists() and fe_test_path.exists():
    train_df = pd.read_parquet(fe_train_path)
    test_df  = pd.read_parquet(fe_test_path)
    print(f'Loaded cached FE: {DATA_VERSION}')
else:
    train_df = engineer_features(train_df)
    test_df  = engineer_features(test_df)
    pq.write_table(pa.Table.from_pandas(train_df, preserve_index=False), fe_train_path)
    pq.write_table(pa.Table.from_pandas(test_df,  preserve_index=False), fe_test_path)
    print(f'Computed and cached FE: {DATA_VERSION}')

Loaded cached FE: fe_v4_native


In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from src.tracking import RUNS_DIR
from src.cv import run_cv_experiment, save_experiment

encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]

cat_features = [c for c in encoded_features if str(X_train[c].dtype) == 'category']
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}  '
      f'features: {len(encoded_features)}  categorical: {len(cat_features)}')

X_train: (594194, 22)  X_test: (254655, 22)  features: 22  categorical: 17


### Fold-contract check

The on-platform row order and `StratifiedKFold(5, shuffle, seed 42)` fold assignment
must match `experiments/cv_folds_seed42.csv.gz` committed from the local environment.
If either assert fires, **stop** — do not save the run.

In [8]:
folds_ref = pd.read_csv('experiments/cv_folds_seed42.csv.gz')   # CWD = repo root

cv_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds_here = np.full(len(train_df), -1)
for fold, (_, va_idx) in enumerate(cv_check.split(np.zeros(len(train_df)), train_df['Churn'])):
    folds_here[va_idx] = fold

assert (folds_ref['id'].to_numpy() == train_df['id'].to_numpy()).all(), \
    'Row order differs from local — OOF would NOT be stackable. Stop.'
assert (folds_ref['fold'].to_numpy() == folds_here).all(), \
    'Fold assignment differs from local — OOF would NOT be stackable. Stop.'
print('Fold contract OK: rows and folds match the committed local assignment.')

Fold contract OK: rows and folds match the committed local assignment.


### Time budget and inner-CV subsample

Kaggle sessions last strictly 12 hours. The Optuna `timeout` parameter stops the study
cleanly at `TUNE_TIMEOUT_S` seconds, leaving enough margin for the 5-fold 2-seed final
refit. Defaults assume baseline timing: ~935 s per full fold (tabm-baseline run) and
~2.5 h for 5 folds × 2 seeds.

The inner-CV subsample is drawn **once** (fixed seed) outside the trial loop so all
trials see identical rows — this keeps the TPE surrogate internally consistent and
prevents the sampler from exploiting lucky subsample draws.

**Smoke-test mode** (`SMOKE_TEST = True`): shrinks the subsample to 5k rows and caps
Optuna at 2 trials so a full end-to-end pass takes a few minutes. Flip back to
`False` for Save & Run All.

In [9]:
import time
from sklearn.model_selection import train_test_split

SMOKE_TEST = False   # set True for a quick end-to-end sanity check

SESSION_LIMIT_S = 12 * 3600          # Kaggle hard limit
REFIT_BUDGET_S  = 2.5 * 3600        # conservative estimate for 5-fold x 2-seed CV
SAFETY_S        = 10 * 60           # buffer before session hard-kill

elapsed = time.time() - NOTEBOOK_T0
TUNE_TIMEOUT_S = SESSION_LIMIT_S - elapsed - REFIT_BUDGET_S - SAFETY_S
print(f'Elapsed so far:  {elapsed/60:.1f} min')
print(f'Optuna timeout:  {TUNE_TIMEOUT_S/3600:.2f} h  ({TUNE_TIMEOUT_S:.0f} s)')
print(f'Refit budget:    {REFIT_BUDGET_S/3600:.1f} h')

# Fixed stratified subsample for inner CV (drawn once; all trials use the same rows).
INNER_N = 5_000 if SMOKE_TEST else 120_000

_, X_inner, _, y_inner = train_test_split(
    X_train, y_train,
    test_size=INNER_N,
    stratify=y_train,
    random_state=42,
)
X_inner = X_inner.reset_index(drop=True)
y_inner = y_inner.reset_index(drop=True)
print(f'Inner-CV subsample: {X_inner.shape}  class balance: {y_inner.mean():.4f}')

Elapsed so far:  1.0 min
Optuna timeout:  9.32 h  (33541 s)
Refit budget:    2.5 h
Inner-CV subsample: (120000, 22)  class balance: 0.2252


### Optuna study

TPE over `lr`, `weight_decay`, and `batch_size`. Each trial:
1. Builds a `TabM_D_Classifier` with the trial's suggested params on `cuda:0`
   (the second GPU is reserved for the final 2-seed refit).
2. Runs 3-fold inner CV on the 120k subsample; objective = mean ROC-AUC.

`SaveBestParamsCallback` writes `best_params.json` after every improvement so
the best configuration survives a mid-trial hard-kill.

**Parameter names.** `TabM_D_Classifier` accepts `lr`, `weight_decay`, and
`batch_size` directly (confirmed against pytabkit 1.7.3). If a future pytabkit
version renames these, adjust the search space dict and the `SeedEnsembleTabM`
wrapper below accordingly.

In [10]:
import json, optuna
from pathlib import Path
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

BEST_PARAMS_PATH = Path('/kaggle/working/best_params.json')
_tune_t0 = None   # set just before optimize() below


class SaveBestParamsCallback:
    """Writes best_params.json whenever a new best trial is found."""
    def __call__(self, study, trial):
        if study.best_trial.number == trial.number:
            payload = {
                'best_value':    study.best_value,
                'best_params':   study.best_params,
                'trial_number':  trial.number,
                'n_trials_done': len(study.trials),
                'elapsed_s':     time.time() - _tune_t0 if _tune_t0 else 0,
            }
            BEST_PARAMS_PATH.write_text(json.dumps(payload, indent=2))
            print(f'  [trial {trial.number}] new best ROC-AUC={study.best_value:.6f}  → saved')


def tabm_objective(trial):
    params = {
        'device':       'cuda:0',   # cuda:1 reserved for the final 2-seed refit
        'random_state': 42,
        'n_epochs':     100,         # fixed max; pytabkit early-stops internally
        'lr':           trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-1, log=True),
        'batch_size':   trial.suggest_categorical('batch_size', [256, 512, 1024, 2048]),
    }
    fold_aucs = []
    for tr_idx, va_idx in inner_cv.split(X_inner, y_inner):
        model = TabM_D_Classifier(**params)
        model.fit(X_inner.iloc[tr_idx], y_inner.iloc[tr_idx])
        va_proba = model.predict_proba(X_inner.iloc[va_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y_inner.iloc[va_idx], va_proba))
    return float(np.mean(fold_aucs))


tabm_study = optuna.create_study(
    study_name='tabm-optuna',
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)

n_trials_cap = 2 if SMOKE_TEST else None   # None = run until timeout

_tune_t0 = time.time()
tabm_study.optimize(
    tabm_objective,
    n_trials=n_trials_cap,
    timeout=None if SMOKE_TEST else TUNE_TIMEOUT_S,
    callbacks=[SaveBestParamsCallback()],
    show_progress_bar=True,
)

elapsed_study = time.time() - _tune_t0
print(f'\nStudy done: {len(tabm_study.trials)} trials in {elapsed_study/3600:.2f} h')
print(f'Best inner-CV ROC-AUC: {tabm_study.best_value:.6f}  (trial {tabm_study.best_trial.number})')
for k, v in tabm_study.best_params.items():
    print(f'  {k:16s} {v}')
print(f'Time remaining: {(SESSION_LIMIT_S - (time.time() - NOTEBOOK_T0))/3600:.2f} h')

   0%|          | 00:00/9:19:00

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12

  [trial 0] new best ROC-AUC=0.913097  → saved


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12

  [trial 6] new best ROC-AUC=0.913214  → saved


...

  [trial 52] new best ROC-AUC=0.913453  → saved


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12

  [trial 87] new best ROC-AUC=0.913460  → saved


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12


Study done: 126 trials in 9.33 h
Best inner-CV ROC-AUC: 0.913460  (trial 87)
  lr               0.004853087769767874
  weight_decay     0.0006359846756374852
  batch_size       256
Time remaining: 2.65 h


In [11]:
# Load best params - works whether the study ran to completion OR was hard-killed.
# If the study cell finished normally, tabm_study is in scope and best_params comes
# from it. If the notebook was killed and re-run from this cell, the JSON fallback
# loads the last-saved improvement.
try:
    best_params = tabm_study.best_params
    best_value  = tabm_study.best_value
    n_trials    = len(tabm_study.trials)
    best_trial  = tabm_study.best_trial.number
    print(f'Using in-memory study best: ROC-AUC={best_value:.6f}')
except NameError:
    if BEST_PARAMS_PATH.exists():
        saved       = json.loads(BEST_PARAMS_PATH.read_text())
        best_params = saved['best_params']
        best_value  = saved['best_value']
        n_trials    = saved['n_trials_done']
        best_trial  = saved['trial_number']
        print(f'Loaded from JSON: ROC-AUC={best_value:.6f}  '
              f'(trial {best_trial}, {n_trials} trials done)')
    else:
        raise FileNotFoundError(
            f'{BEST_PARAMS_PATH} not found — did the study run at least one trial?')

print('Best hyperparameters:')
for k, v in best_params.items():
    print(f'  {k:16s} {v}')

Using in-memory study best: ROC-AUC=0.913460
Best hyperparameters:
  lr               0.004853087769767874
  weight_decay     0.0006359846756374852
  batch_size       256


### 2-seed ensemble wrapper with tuned hyperparameters

Same architecture as `predict-customer-churn-tabm-gpu-min3.ipynb`: one TabM per GPU
trained concurrently (seeds 42/43), full fold-training partition per model,
predictions averaged. The tuned `lr`, `weight_decay`, and `batch_size` are wired
into the constructor and forwarded to each `TabM_D_Classifier` instance.

`lr=None` / `weight_decay=None` fall back to pytabkit's calibrated defaults, so the
wrapper stays functional if a parameter was excluded from the search space.

In [12]:
from concurrent.futures import ThreadPoolExecutor
from sklearn.base import BaseEstimator, ClassifierMixin


class SeedEnsembleTabM(BaseEstimator, ClassifierMixin):
    """TabM 2-seed deep ensemble: one model per GPU, concurrent fits, averaged predictions."""

    def __init__(self, n_epochs=100, batch_size=1024, lr=None, weight_decay=None,
                 random_state=42, devices=('cuda:0', 'cuda:1')):
        self.n_epochs     = n_epochs
        self.batch_size   = batch_size
        self.lr           = lr
        self.weight_decay = weight_decay
        self.random_state = random_state
        self.devices      = devices

    def _build_model(self, dev, seed):
        kw = {'device': dev, 'random_state': seed, 'n_epochs': self.n_epochs,
              'batch_size': self.batch_size}
        if self.lr is not None:
            kw['lr'] = self.lr
        if self.weight_decay is not None:
            kw['weight_decay'] = self.weight_decay
        return TabM_D_Classifier(**kw)

    def fit(self, X, y):
        X = X.reset_index(drop=True)
        y = y.reset_index(drop=True)

        def fit_one(arg):
            i, dev = arg
            model = self._build_model(dev, self.random_state + i)
            model.fit(X, y)
            return model

        with ThreadPoolExecutor(max_workers=len(self.devices)) as pool:
            self.models_ = list(pool.map(fit_one, enumerate(self.devices)))
        self.classes_ = self.models_[0].classes_
        return self

    def predict_proba(self, X):
        with ThreadPoolExecutor(max_workers=len(self.devices)) as pool:
            probas = list(pool.map(lambda m: m.predict_proba(X), self.models_))
        return np.mean(probas, axis=0)

In [13]:
from importlib.metadata import version

DEVICES = tuple(f'cuda:{i}' for i in range(max(1, torch.cuda.device_count())))

run_params = {
    'n_epochs':     100,
    'batch_size':   best_params['batch_size'],
    'lr':           best_params['lr'],
    'weight_decay': best_params['weight_decay'],
    'random_state': 42,
    'devices':      DEVICES,
}
print('Run params:')
for k, v in run_params.items():
    print(f'  {k:16s} {v}')

study_summary = (
    f'Optuna TPE {n_trials} trials, 3-fold inner CV, 120k-row subsample, '
    f'ROC-AUC {best_value:.6f} at trial {best_trial}.'
)

run_config = {
    'model_factory': lambda params: SeedEnsembleTabM(**params),
    'params':        run_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'tabm-optuna-fe-min3',
    'notes': (
        'TabM (pytabkit TabM_D_Classifier) Optuna-tuned 2-seed ensemble on Kaggle T4 x2. '
        f'{study_summary} '
        'FE: fe_v4_native (contract_x_internet + contract_x_payment + AverageMonthly). '
        'Fold assignment asserted against experiments/cv_folds_seed42.csv.gz. '
        f'pytabkit={version("pytabkit")}, torch={version("torch")}. '
        'Data regenerated on-platform; GPU run not bit-reproducible. '
        'Notebook: kaggle/predict-customer-churn-tabm-optuna.ipynb.'
    ),
    'parent_run_id': '20260601-022539-10bd38',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

Run params:
  n_epochs         100
  batch_size       256
  lr               0.004853087769767874
  weight_decay     0.0006359846756374852
  random_state     42
  devices          ('cuda:0', 'cuda:1')


In [14]:
# Step 1 - Run the experiment.
# SMOKE-TEST FIRST (if not already done via SMOKE_TEST=True in the budget cell):
# uncomment the slice below, run one fold, confirm both GPUs show load in nvidia-smi,
# then restore for the full run. Keep n_splits=5 - the fold contract requires it.
# result = run_cv_experiment(run_config, X_train.head(20_000), y_train.head(20_000),X_test.head(10_000), encoded_features)
result = run_cv_experiment(run_config, X_train, y_train, X_test, encoded_features)

Run ID: 20260612-181438-3a3dd2
Tag:    tabm-optuna-fe-min3



/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12

Fold 0: accuracy=0.8577  roc_auc=0.9132  (fit 597.0s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12

Fold 1: accuracy=0.8588  roc_auc=0.9147  (fit 835.3s)


...

Fold 2: accuracy=0.8582  roc_auc=0.9137  (fit 579.4s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12

Fold 3: accuracy=0.8595  roc_auc=0.9149  (fit 823.3s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12

Fold 4: accuracy=0.8583  roc_auc=0.9123  (fit 695.4s)

OOF accuracy: 0.8585
OOF ROC-AUC:  0.9137
Folds:        0.8585 ± 0.0006

Run complete. Call save_experiment(result) to log this run permanently.


In [15]:
# Step 2 - Save the run. Review the OOF ROC-AUC printed above first, and only save
# if the fold-contract cell passed.
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260612-181438-3a3dd2


### Build a submission (optional)

`test_proba_mean` is the fold-bagged (5 folds × 2 seeds) churn probability for the
full test set. Submit the probability directly (competition metric is ROC-AUC).

In [16]:
# submission = pd.DataFrame({
#     'id':    test_df['id'],
#     'Churn': result['artifacts']['test_proba_mean'],
# })
# submission.to_csv('/kaggle/working/submission.csv', index=False)
# print(submission.head())
# print('wrote /kaggle/working/submission.csv', submission.shape)

### Bundle run artifacts into one zip

Copies the run directory, `runs.csv`, `best_params.json`, and the source notebook
into a single archive on the Output tab. Follow **§7–8 of
`docs/kaggle_gpu_workflow.md`** to merge back into the local repo.

In [17]:
import shutil
from pathlib import Path
from src.tracking import RUNS_DIR, RUNS_CSV

BUNDLE = Path('/kaggle/working/bundle')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)

# 1) run artifacts (params, oof_proba, test_proba_*, metrics, env, git diff)
shutil.copytree(RUNS_DIR / run_id, BUNDLE / 'runs' / run_id)
# 2) master index row
shutil.copy(RUNS_CSV, BUNDLE / 'runs.csv')
# 3) best_params.json (Optuna study result, independent of run logging)
if BEST_PARAMS_PATH.exists():
    shutil.copy(BEST_PARAMS_PATH, BUNDLE / 'best_params.json')
# 4) source notebook from the cloned repo
src_nb = Path(REPO_ROOT) / 'kaggle' / 'predict-customer-churn-tabm-optuna.ipynb'
if src_nb.exists():
    shutil.copy(src_nb, BUNDLE / src_nb.name)

archive = shutil.make_archive(f'/kaggle/working/{run_id}_bundle', 'zip', BUNDLE)
print('wrote', archive)

wrote /kaggle/working/20260612-181438-3a3dd2_bundle.zip
